# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata using the .metadata attribute
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}:")
print(metadata_json['description'])

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns.

In [ ]:
# Display available record sets, fields, and columns (referenced by @id)
all_record_sets = dataset.metadata.record_sets
print("Available record sets (and their @id):")
for rs in all_record_sets:
    print(f"  - {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    print("    Fields and columns in this record set:")
    for field in rs.get('field', []):
        field_obj = None
        # Find full field object by @id
        for f in dataset.metadata.fields:
            if f['@id'] == field:
                field_obj = f
                break
        if field_obj:
            print(f"      * {field_obj['@id']} | Field name: {field_obj.get('name', 'N/A')} | Data type: {field_obj.get('dataType', 'N/A')}")
            # Print columns for this field
            for col in field_obj.get('column', []):
                print(f"         - column @id: {col}")
    print()
print("\nTo load records from a record set, use its @id (e.g. the value printed above). Below is a sample record from one record set:")
if all_record_sets:
    example_rs_id = all_record_sets[0]['@id']
    try:
        recs = dataset.records(record_set=example_rs_id)
        print(next(recs))
    except Exception as e:
        print(f"Unable to print records for {example_rs_id}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the `@id` of record sets and fields as referenced above.

In [ ]:
# Prepare list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for record set {rs_id}")
        else:
            print(f"No records found for record set {rs_id}")
    except Exception as e:
        print(f"Error loading records for record set {rs_id}: {e}")

# List columns in the main data record set (select first available with records)
main_record_set_id = None
main_df = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        main_df = df
        break

if main_record_set_id:
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(main_df.columns.tolist())
    display(main_df.head())
else:
    print("No non-empty record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA operations such as filtering, normalization, and grouping to the fields in the main record set. All fields are referenced by their `@id`s and column names.

In [ ]:
import numpy as np

if main_record_set_id and main_df is not None and not main_df.empty:
    # Try to guess a numeric field (@id) from the columns
    numeric_fields = main_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Choose the first numeric column for demonstration
        print(f"Analyzing numeric field: '{numeric_field_id}'\n")
        threshold = main_df[numeric_field_id].quantile(0.80)  # Use 80th percentile as an example
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical field
        likely_group_fields = [col for col in main_df.columns if main_df[col].dtype == object]
        if likely_group_fields:
            group_field = likely_group_fields[0]
            print(f"\nGrouping normalized values by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[f"{numeric_field_id}_normalized"].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("Main data set is not available or empty.")

## 5. Visualization
Visualize data distributions or relationships in the dataset. This example shows a histogram and a boxplot for a selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_df is not None and not main_df.empty and numeric_fields:
    field = numeric_fields[0]
    plt.figure(figsize=(10, 4))
    sns.histplot(main_df[field].dropna(), kde=True)
    plt.title(f"Distribution of {field}")
    plt.xlabel(field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if likely_group_fields:
        group_field = likely_group_fields[0]
        plt.figure(figsize=(12, 4))
        sns.boxplot(x=group_field, y=field, data=main_df)
        plt.title(f"Boxplot of {field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, you explored the FAIR^2 dataset defined by a Croissant schema, loading data using `mlcroissant`. You viewed data structure by `@id`, extracted and visualized records, and performed preliminary EDA on selected numeric fields. Adjust the record set and field `@id`s as needed for your specific analyses.